## Choose the native HHL program


## Spectroscopy guides a classical table reduction

The visible probe builder prepares the input and composes four-bit QPE for
the rebuilt portfolio operator. FlagQuantum constructs the native program;
the host reads finite-shot spectral counts, selects phase words and constructs
a three-bit reciprocal table. That table reduction is an explicit classical
algorithm, not an automatic synth approximation.

Compare full and reduced tables, indicator/test joint counts and conditional
overlap together. The width and depth profiles use the same declared source
model and separate CX/U3 resource conversion. Frequent phase words are observed
samples, not exact eigenvalues. Dropped data and phase registers remain private
environment. A conditional SWAP overlap is not a classical solution readout.


In [ ]:
from flagquantum import synth
from flagquantum.synth import Output
from flagquantum.synth import QArray
from flagquantum.synth import QBit
from flagquantum.synth import QNum
from flagquantum.synth import allocate
from flagquantum.synth import prepare_amplitudes
from flagquantum.synth import qfunc
from flagquantum.synth import qpe
from flagquantum.synth import unitary
from typing import Annotated

from examples.synthesis.applications.finance.hybrid_hhl_for_portfolio_optimization.portfolio_optimization_with_hhl.overview.quantum import (
    Model,
)


def spectroscopy(model: Model) -> synth.Program:
    evolution = tuple(tuple(map(complex, row)) for row in model.evolution)
    amplitudes = tuple(map(float, model.normalized_input))

    @qfunc
    def probe(
        amplitudes: list[float],
        phase: Output[Annotated[QNum, (4, False, 4)]],
        data: Output[QArray[QBit]],
    ) -> None:
        allocate(phase)
        prepare_amplitudes(amplitudes, 0, data)
        qpe(lambda: unitary(evolution, data), phase)

    return synth.synthesize(
        probe,
        classical_inputs={"amplitudes": amplitudes},
        max_qubits=12,
        max_workspace=0,
        memory_limit_bytes=1024**2,
    )


from IPython.display import Image, display

from examples.synthesis.applications.finance.hybrid_hhl_for_portfolio_optimization.portfolio_optimization_with_hhl.overview.main import (
    report,
    save_figures,
)
from examples.synthesis.applications.finance.hybrid_hhl_for_portfolio_optimization.portfolio_optimization_with_hhl.overview.model import (
    PHASE_BITS,
    REDUCED_BITS,
    SHOTS,
    build_model,
    reduced_table,
)
from examples.synthesis.applications.finance.hybrid_hhl_for_portfolio_optimization.portfolio_optimization_with_hhl.overview.workflow import (
    run_experiment,
)


## Resolve phase words to reciprocal rotations


In [ ]:
print("Full and reduced table sizes:", 1 << PHASE_BITS, 1 << REDUCED_BITS)


## Recover the two-asset linear system


In [ ]:
model = build_model()
print("Portfolio operator:", model.operator)


## Normalize the original forcing


In [ ]:
print("Normalized input:", model.normalized_input)


## Compute the exact spectrum classically


In [ ]:
print("Exact eigenvalues:", model.eigenvalues)


## Set the full phase counter width


In [ ]:
print("Phase bits:", PHASE_BITS)


## Scale, shift and establish a reciprocal cutoff


In [ ]:
print("Shift, scale, cutoff:", model.eigenvalue_bound, model.scale, model.cutoff)


## Regenerate the evolution unitary


In [ ]:
print("Unitary shape:", tuple(model.evolution.shape))


## Create the native spectroscopy program


In [ ]:
probe = spectroscopy(model)
print("Probe phase and data ports:", probe.registers)


## Run spectroscopy and the three HHL profiles


In [ ]:
model, spectrum, profiles = run_experiment()
print("Measured phase counts:", spectrum.counts)


## Inspect compiled spectroscopy resources


In [ ]:
print(probe.report["resources"])


## Recover eigenvalue estimates from samples


In [ ]:
print(
    "Estimated eigenvalues and inverse weights:",
    tuple(zip(spectrum.eigenvalues, spectrum.projections, strict=True)),
)


## Plot the full measured spectrum


In [ ]:
figures = save_figures(model, spectrum)
display(Image(filename=str(figures[0])))


## Identify the four high-probability rows


In [ ]:
print("Four most frequent phase words:", spectrum.words[:4])


## Choose the two feed-forward rotations


In [ ]:
print("Selected phase words:", spectrum.selected_words)


## Truncate the phase resolution by one bit


In [ ]:
print("Reduced phase bits:", REDUCED_BITS)


## Compare selected bins and exact eigenvalues


In [ ]:
display(Image(filename=str(figures[1])))


## Solve the four-dimensional system directly


In [ ]:
print("Normalized direct solution:", model.comparison)


## Build the reduced reciprocal table


In [ ]:
reduced = reduced_table(model, spectrum.selected_words)
print("Three-bit rotation table:", reduced)


## Review the depth profile ceiling


In [ ]:
assert profiles[0].resources.n_qubits <= 12
print("Measured depth-profile width:", profiles[0].resources.n_qubits)


## Review the local execution and compiler settings


In [ ]:
print("Local CPU, fixed finite-shot seeds, separate CX/U3 resource conversion")


## Record the original finite shot budget


In [ ]:
assert sum(spectrum.counts) == SHOTS
print("Shots per histogram:", SHOTS)


## Read measured feed-forward SWAP counts


In [ ]:
print("Joint indicator and test counts:", profiles[0].counts)


## Inspect compiled feed-forward resources


In [ ]:
print(profiles[0].resources)


## Measure overlap on the indicator-one branch


In [ ]:
print("Hybrid HHL conditional overlap:", profiles[0].overlap)


## Compare the synthesized width profile


In [ ]:
print("Width objective resources:", profiles[1].resources)


## Evaluate the four-bit HHL baseline


In [ ]:
print("Full four-bit table:", model.basic_angles)
print("Basic HHL conditional overlap:", profiles[2].overlap)


## Collect the converted gate counts


In [ ]:
for row in profiles:
    print(row.name, row.resources.operation_counts, row.target_identity)


## Report width, depth, CX count and SWAP overlap


In [ ]:
report(spectrum, profiles)
